# Right Agent Group — full stack + REAL CALL test on Kaggle GPU

**Before running:** Settings (right panel) → Accelerator = **GPU T4 x2** → Internet = **ON** → attach the private dataset **rag-project**.

Then **Run All**. Boot ≈ 10–15 min (mostly the AI model download).

What you get:
1. The full website on a public test URL (dashboard, analytics, everything)
2. A public WebSocket URL for the Exotel Voicebot applet → a **real phone call to your verified number** with Priya talking, GPU-fast

⚠️ Test environment only: URLs change every run, everything is wiped at session end (12h max).

In [ ]:
%%bash
# ---- 1. System dependencies: Node 22, PostgreSQL, ffmpeg, cloudflared ----
set -e
curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
apt-get install -y -qq nodejs postgresql ffmpeg > /dev/null 2>&1
curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
echo "node: $(node -v) | psql: $(psql --version | awk '{print $3}') | ffmpeg + cloudflared OK"

In [ ]:
%%bash
# ---- 2. Copy project from the dataset + start PostgreSQL with matching password ----
set -e
cp -r /kaggle/input/rag-project/project /kaggle/working/app
# restore Next.js dynamic-route dirs that were bracket-encoded for the Kaggle upload
python3 - <<PYEOF
import os
for root, dirs, files in os.walk("/kaggle/working/app", topdown=False):
    for name in dirs + files:
        if "__LB__" in name:
            os.rename(os.path.join(root, name), os.path.join(root, name.replace("__LB__", "[").replace("__RB__", "]")))
PYEOF
cd /kaggle/working/app
PGPASS=$(grep -E '^PG_PASSWORD=' .env | cut -d= -f2 | awk '{print $1}')
service postgresql start > /dev/null
sudo -u postgres psql -c "ALTER USER postgres PASSWORD '${PGPASS}';" > /dev/null
echo "PostgreSQL running, password synced with .env"

In [ ]:
%%bash
# ---- 3. Ollama on GPU + pull the model (~5 GB, the slow step) ----
set -e
curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
nohup ollama serve > /kaggle/working/ollama.log 2>&1 &
sleep 5
ollama pull llama3.1:8b
echo "--- GPU check ---"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
# ---- 4. Install app dependencies (root + voicebot + STT) ----
set -e
cd /kaggle/working/app
npm install --silent 2>&1 | tail -1
cd server && npm install --silent 2>&1 | tail -1 && cd ..
pip install -q -r server/stt-service/requirements.txt
echo "All dependencies installed"

In [ ]:
# ---- 5. TWO public tunnels: website (3000) + voicebot WebSocket (3002) ----
import subprocess, re

def quick_tunnel(port):
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(90):
        line = p.stdout.readline()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m: return m.group(0)
    raise RuntimeError(f"tunnel for port {port} failed — re-run this cell")

web_url = quick_tunnel(3000)
vb_url  = quick_tunnel(3002)
wss_url = vb_url.replace("https://", "wss://") + "/voicebot"

open("/kaggle/working/tunnel_url.txt", "w").write(web_url)
open("/kaggle/working/wss_url.txt", "w").write(wss_url)
print("WEBSITE URL :", web_url)
print("VOICEBOT WSS:", wss_url)

In [ ]:
%%bash
# ---- 6. Point .env at the tunnel + GPU settings, then create the database ----
set -e
cd /kaggle/working/app
URL=$(cat /kaggle/working/tunnel_url.txt)
sed -i "s|^NEXT_PUBLIC_APP_URL=.*|NEXT_PUBLIC_APP_URL=${URL}|" .env
sed -i "s|^OLLAMA_GPU=.*|OLLAMA_GPU=true|" .env
sed -i "s|^STT_FORCE_DEVICE=.*|STT_FORCE_DEVICE=cuda|" .env
grep -q '^APP_INTERNAL_URL=' .env || echo 'APP_INTERNAL_URL=http://127.0.0.1:3000' >> .env
npm run db:setup && npm run db:check | tail -3

In [ ]:
%%bash
# ---- 7. Build + start everything (website, STT on GPU, voicebot) ----
cd /kaggle/working/app
npm run build 2>&1 | tail -3
nohup npm run start   > /kaggle/working/web.log      2>&1 &
cd server/stt-service
nohup python -m uvicorn app:app --host 127.0.0.1 --port 3003 > /kaggle/working/stt.log 2>&1 &
cd ..
nohup node voicebot-server.js > /kaggle/working/voicebot.log 2>&1 &
sleep 20
echo '--- listening ports (want 3000, 3002, 3003, 11434) ---'
ss -tlnp | grep -E ':(3000|3002|3003|11434)' | awk '{print $4}'

In [ ]:
# ---- 8. Login cookie + ALL your URLs and next steps ----
import base64, hashlib, hmac, json, time, re

env = open("/kaggle/working/app/.env", encoding="utf-8").read()
secret = re.search(r"^AUTH_SECRET=(\S+)", env, re.M).group(1)
b64 = lambda b: base64.urlsafe_b64encode(b).rstrip(b"=").decode()
payload = b64(json.dumps({"email": "kaggle-test@rightagentgroup.local", "exp": int(time.time()) + 86400}).encode())
sig = b64(hmac.new(secret.encode(), payload.encode(), hashlib.sha256).digest())

web = open("/kaggle/working/tunnel_url.txt").read().strip()
wss = open("/kaggle/working/wss_url.txt").read().strip()
print("=" * 60)
print("1. OPEN THE WEBSITE :", web)
print("   Press F12 -> Console -> paste this line -> Enter:")
print(f'   document.cookie="rag_session={payload}.{sig}; path=/"')
print("   Then go to:", web + "/dashboard")
print("=" * 60)
print("2. EXOTEL SETUP (once per Kaggle session):")
print("   my.exotel.com -> App Bazaar -> your flow (1288523)")
print("   -> Voicebot applet -> set URL to:")
print("  ", wss)
print("   -> Save the flow.")
print("=" * 60)
print("3. Then run the last cell to place the real test call.")

In [ ]:
%%bash
# ---- 9. Health + GPU speed test (no phone call yet) ----
cd /kaggle/working/app
KEY=$(grep -E '^WHATSAPP_SERVICE_KEY=' .env | cut -d= -f2 | awk '{print $1}')
echo '--- STT (expect model large-v3, device cuda) ---'
curl -s -H "x-api-key: $KEY" http://127.0.0.1:3003/health
echo ''
echo '--- Priya brain speed on GPU (was 10-25s on the laptop) ---'
time curl -s -X POST http://127.0.0.1:3000/api/calls/turn -H 'Content-Type: application/json' \
  -H "x-api-key: $KEY" -d '{"event":"turn","callSid":"KAGGLE-TEST-1","speech":"hello, I want a home loan","language":"english"}'

In [ ]:
# ---- 10. REAL PHONE CALL — read before running! ----
# Priya will actually dial the number below. Requirements:
#   - Exotel KYC approved (trial: the number must be VERIFIED in Exotel)
#   - Cell 8 step 2 done (wss URL saved in the Exotel flow THIS session)
# Uses trial credits. Change CONFIRM to True, then run.

CONFIRM = False
PHONE   = "+919908838090"   # your verified test number

import re, urllib.request, json as j
if not CONFIRM:
    print("Not calling. Set CONFIRM = True (after doing cell 8 step 2) and run again.")
else:
    env = open("/kaggle/working/app/.env", encoding="utf-8").read()
    key = re.search(r"^WHATSAPP_SERVICE_KEY=(\S+)", env, re.M).group(1)
    # go through the app's own outbound route so the call is fully tracked
    import base64, hashlib, hmac, time
    secret = re.search(r"^AUTH_SECRET=(\S+)", env, re.M).group(1)
    b64 = lambda b: base64.urlsafe_b64encode(b).rstrip(b"=").decode()
    p = b64(j.dumps({"email": "kaggle-test@rightagentgroup.local", "exp": int(time.time()) + 3600}).encode())
    s = b64(hmac.new(secret.encode(), p.encode(), hashlib.sha256).digest())
    req = urllib.request.Request(
        "http://127.0.0.1:3000/api/calls",
        data=j.dumps({"phone": PHONE, "language": "telugu"}).encode(),
        headers={"Content-Type": "application/json", "Cookie": f"rag_session={p}.{s}"},
        method="POST",
    )
    try:
        print(urllib.request.urlopen(req, timeout=60).read().decode())
        print("\nPHONE SHOULD RING NOW. Answer it and talk to Priya!")
        print("Afterwards: check Voice Logs in the dashboard for the recording + transcript.")
    except urllib.error.HTTPError as e:
        print("Call failed:", e.read().decode())